# Grove — Mandarin Auto-Labeling (Kaggle GPU)

**One Run-All produces a labeled dataset.** Click `Run All`, wait, then download the single
`grove_dataset.zip` from the output panel.

Grounding DINO is a *teacher* that drafts labels (SPEC.md §2). The real deliverable is the
**coordinate files** (YOLO + COCO), not the preview images. After downloading, review/correct the
boxes locally with `grove review` before training anything (human review is mandatory).

**The ONLY thing you may need to edit is the input path in the Config cell.**

## 1. Preflight — GPU + internet check
Grounding DINO needs a GPU to run at folder scale, and first-run weight download needs internet.
This cell only *checks* and tells you exactly what to flip — it cannot toggle the sidebar itself.

In [ ]:
import sys, socket

ok = True

# --- GPU ---
try:
    import torch
    if torch.cuda.is_available():
        print(f"[OK] GPU: {torch.cuda.get_device_name(0)}")
    else:
        ok = False
        print("[FAIL] No CUDA GPU. FIX: Notebook sidebar -> Accelerator -> GPU (T4 or P100).")
except Exception as e:
    ok = False
    print(f"[FAIL] torch not importable ({e}). FIX: Notebook sidebar -> Accelerator -> GPU (T4 or P100).")

# --- Internet (needed to pip install grove + auto-download detector weights) ---
try:
    socket.create_connection(("pypi.org", 443), timeout=5).close()
    print("[OK] Internet is on.")
except Exception:
    ok = False
    print("[FAIL] No internet. FIX: Notebook sidebar -> Internet -> On.")

if not ok:
    # Stop the Run-All here so later cells don't fail confusingly.
    raise SystemExit("Preflight failed — fix the sidebar settings above, then Run All again.")
print("\nPreflight passed.")

## 2. Install Grove + detector
Installs Grove plus a pure-Python **Grounding DINO** (HuggingFace `transformers`) that runs on the
GPU with no compiling — the same model as the autodistill backend, without its brittle native build.
Detector **weights download automatically on first `detect`** — you never fetch them by hand.

In [ ]:
import os, subprocess, sys, glob, shutil, importlib

# === SET THIS ONCE (optional) =================================================
# Paste your repo URL here after you push to GitHub. Then anyone running this
# notebook needs ZERO edits — Run All just works. Leave it "" if you instead
# attach the repo as a Kaggle Dataset (the notebook auto-finds it either way).
GROVE_REPO_URL = "https://github.com/leeohwang/mandarin_tree_detector.git"   # e.g. "https://github.com/you/mandarin_tree_detector.git"
# =============================================================================
GROVE_REPO_URL = os.environ.get("GROVE_REPO_URL", GROVE_REPO_URL).strip()

# --- Locate the Grove repo on Kaggle --------------------------------------
# Priority: GROVE_REPO_DIR env var -> a working checkout at /kaggle/working/grove
# -> an attached Kaggle Dataset (any folder under /kaggle/input containing
# pyproject.toml) -> git clone GROVE_REPO_URL. Strangers do nothing: attach the
# repo as a Dataset, or rely on the URL you baked in above.
def _find_repo():
    cand = os.environ.get("GROVE_REPO_DIR")
    if cand and os.path.isfile(os.path.join(cand, "pyproject.toml")):
        return cand
    if os.path.isfile("/kaggle/working/grove/pyproject.toml"):
        return "/kaggle/working/grove"
    # Recursive search: a Kaggle Dataset can nest the repo at any depth (or wrap it
    # in an extra folder), so don't assume a fixed depth. Pick the SHALLOWEST hit
    # (the top-level repo pyproject, not one inside a subpackage).
    hits = sorted(glob.glob("/kaggle/input/**/pyproject.toml", recursive=True), key=len)
    if hits:
        return os.path.dirname(hits[0])
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None and GROVE_REPO_URL:
    # No attached checkout, but a URL was provided -> clone it (needs Internet On).
    REPO_DIR = "/kaggle/working/grove"
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    print(f"Cloning {GROVE_REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", GROVE_REPO_URL, REPO_DIR])
if REPO_DIR is None:
    REPO_DIR = "/kaggle/working/grove"
    print("[WARN] Grove repo not found. Do ONE of:")
    print("       (a) attach the repo as a Kaggle Dataset (Add Data panel), or")
    print("       (b) set GROVE_REPO_URL at the top of this cell to your GitHub URL.")

# /kaggle/input is READ-ONLY, but an editable (-e) install must write *.egg-info into
# the source dir. If the repo lives under /kaggle/input, copy it somewhere writable.
if REPO_DIR.startswith("/kaggle/input"):
    dst = "/kaggle/working/grove"
    if os.path.abspath(REPO_DIR) != os.path.abspath(dst):
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(REPO_DIR, dst)
    REPO_DIR = dst
print("Using REPO_DIR:", REPO_DIR)

# --- Install grove + a detector backend ---------------------------------------
# Run pip FROM the repo dir with the canonical "-e ." form (pip rejects an absolute
# path + extras like `-e "/abs/path[gpu]"`). grove core first (light, always builds),
# then the detector stack.
def _pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", *args],
                          cwd=REPO_DIR, capture_output=True, text=True)

r = _pip("-q", "-e", ".")
if r.returncode:
    print(r.stdout[-800:]); print(r.stderr[-2000:]); raise SystemExit("grove core install failed")

# Detector: pure-Python Grounding DINO via HuggingFace transformers. Same IDEA-Research
# model as the autodistill backend, but plain wheels (NO C++/CUDA compile), running on
# the GPU. Used on Kaggle because the autodistill native build (and the
# autodistill-yolo-world fallback) are brittle on Kaggle's current Python 3.12 image.
# torch/torchvision already ship on Kaggle, so we do NOT reinstall them.
print("Installing detector stack (pure-Python Grounding DINO via transformers)...")
r = _pip("-q", "transformers>=4.40,<5", "huggingface_hub>=0.23,<1", "timm>=0.9",
         "safetensors>=0.4", "supervision>=0.22,<0.27", "opencv-python-headless>=4.8,<5")
if r.returncode:
    print(r.stdout[-800:]); print(r.stderr[-2000:]); raise SystemExit("detector install failed")
DETECTOR_BACKEND = "grounding_dino_hf"
print("Grounding DINO (HF transformers) ready — GPU teacher/labeler.")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
import grove
print("grove", grove.__version__, "installed.")


## 3. Config
Loads the repo's committed **`config.example.yaml`** (the single source of truth for prompt,
thresholds, classes and split) and overrides only the paths — so this GPU run and the local review
step can never disagree about what is being detected. The shipped default detects **individual
trees** (`"tree trunk" -> tree`); change `config.example.yaml` to retarget both halves at once.

**Edit ONE line (only if needed):** `INPUT_DIR` — it auto-detects the uploaded image folder; set it
by hand only if the guess is wrong. Outputs go under `/kaggle/working/data` and are auto-created.

In [ ]:
import os
from pathlib import Path
import yaml

# --- Auto-detect the uploaded image folder --------------------------------
# Picks the folder under /kaggle/input with the most images, so there is NOTHING
# to edit by hand. Override with the GROVE_INPUT_DIR env var (or edit INPUT_DIR)
# only if it guesses wrong.
_IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def _autodetect_input():
    base = Path("/kaggle/input")
    if not base.exists():
        return None
    best, best_n = None, 0
    for root, _dirs, files in os.walk(base):
        n = sum(1 for f in files if Path(f).suffix.lower() in _IMG_EXTS)
        if n > best_n:
            best, best_n = root, n
    return best

INPUT_DIR = os.environ.get("GROVE_INPUT_DIR") or _autodetect_input() or "/kaggle/input/your-images"
WORK_DIR = "/kaggle/working/data/work"
EXPORT_DIR = "/kaggle/working/data/dataset"
print("Using INPUT_DIR:", INPUT_DIR)

# --- SINGLE SOURCE OF TRUTH ------------------------------------------------
# Start from the repo's COMMITTED config.example.yaml so this GPU run uses EXACTLY
# the same detector ontology / thresholds / classes as the local review path.
# We override ONLY the paths (Kaggle mounts /kaggle/input read-only). This is what
# keeps the two halves from silently disagreeing about what is being detected —
# edit config.example.yaml once and BOTH the Kaggle run and local review follow.
example_path = Path(REPO_DIR) / "config.example.yaml"
cfg_dict = yaml.safe_load(example_path.read_text())
cfg_dict["paths"] = {"input_dir": INPUT_DIR, "work_dir": WORK_DIR, "export_dir": EXPORT_DIR}

CONFIG_PATH = "/kaggle/working/config.yaml"
Path(CONFIG_PATH).write_text(yaml.safe_dump(cfg_dict, sort_keys=False))

from grove.core.config import load_config
cfg = load_config(CONFIG_PATH)   # validates + auto-creates work_dir & export_dir

# Use whichever backend the install cell selected (HF Grounding DINO on Kaggle).
import importlib.util as _u
# Prefer the backend the install cell set; fall back to probing installed packages
# if this cell is somehow run standalone.
cfg.detector.backend = globals().get("DETECTOR_BACKEND") or (
    "grounding_dino_hf" if _u.find_spec("transformers") else "grounding_dino")
print("Detector backend:", cfg.detector.backend,
      "| classes:", cfg.classes,
      "| ontology:", cfg.detector.ontology)
n = (sum(1 for p in Path(cfg.paths.input_dir).rglob("*") if p.suffix.lower() in _IMG_EXTS)
     if Path(cfg.paths.input_dir).exists() else 0)
print(f"Config OK. Found ~{n} image(s) in {cfg.paths.input_dir}."
      + ("  [WARN] 0 found — set GROVE_INPUT_DIR or edit INPUT_DIR." if n == 0 else ""))


## 4. Run the pipeline: ingest → detect → annotate → export
`ingest` normalizes EXIF + builds the manifest; `detect` runs Grounding DINO (GPU, resumable);
`annotate` draws QC previews; `export` writes the draft YOLO + COCO dataset — **the real output**.

In [ ]:
from grove.pipeline.ingest import ingest
from grove.pipeline.detect import detect
from grove.pipeline.annotate import annotate
from grove.pipeline.export import export

manifest = ingest(cfg)
print(f"ingest:   {len(manifest.images)} image(s) -> {WORK_DIR}/manifest.json")

pred = detect(cfg)   # downloads detector weights on first run; resumable across sessions
total_boxes = sum(len(r.detections) for r in pred.images)
print(f"detect:   {total_boxes} box(es) across {len(pred.images)} image(s) -> {WORK_DIR}/predictions.json")

annotate(cfg)
print(f"annotate: QC previews -> {WORK_DIR}/previews/")

summary = export(cfg)
print("export:  ", summary)

## 5. Sanity check — boxes on the detected class ("it works")
Shows a few annotated previews inline (trees by default). These are for **your eyes only** — the
dataset coordinate files are the deliverable. Expect misses/false positives; that is exactly what
the local review step fixes.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

preview_dir = Path(WORK_DIR) / "previews"
previews = sorted([p for p in preview_dir.glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}])[:6]

if not previews:
    print("No previews found — check that detect produced boxes.")
else:
    cols = min(3, len(previews))
    rows = (len(previews) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = [axes] if rows * cols == 1 else axes.ravel()
    for ax in axes:
        ax.axis('off')
    for ax, p in zip(axes, previews):
        ax.imshow(plt.imread(str(p)))
        ax.set_title(p.name, fontsize=9)
    plt.tight_layout()
    plt.show()

## 6. Package the single download artifact
Zips everything under `/kaggle/working/data` (work/ = prepared images + manifest + predictions,
dataset/ = draft YOLO+COCO) into **one** `grove_dataset.zip`. Download it from the output panel,
unpack locally, then run `grove review`.

In [ ]:
import shutil, os

DATA_DIR = "/kaggle/working/data"
ZIP_BASE = "/kaggle/working/grove_dataset"   # .zip appended by make_archive

zip_path = shutil.make_archive(ZIP_BASE, "zip", root_dir=DATA_DIR)
size_mb = os.path.getsize(zip_path) / 1e6
print(f"Dataset packaged: {zip_path}  ({size_mb:.1f} MB)")
print("\nDOWNLOAD: right panel -> Output -> grove_dataset.zip  (single artifact).")
print("Next: unpack it where your local config points, then run `grove review` to correct boxes.")